# Trend Tracker — Notebook 03: Insights Generation

Builds feature matrices and runs all Phase 1 analytical work on the clean
corpus from `01_preprocess.ipynb`. `make_vec`, `add_bin`, and `group_key` —
all imported from `utils.py` — are shared across Steps 1, 3, and 4, which is
the main reason those three live together.

| Step | What | Key output |
|------|------|------------|
| 1 | Build TF-IDF matrices | `features/X_*.npz`, `vocab/vocab_*.csv` |
| 2 | Quality CP2 | `quality/quality_cp2.json` |
| 3 | Category TF-IDF | `analysis/category_tfidf.csv` |
| 4 | NMF topic discovery | `analysis/nmf_topics.csv`, `nmf_weights.csv` |
| 5 | LLM topic labeling | `analysis/llm_topic_labels.json` |


## Setup

In [1]:
import sys, json, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

import pandas as pd
import numpy as np

# Make local modules importable when the notebook is run from the project root.
ROOT = Path(".")
sys.path.insert(0, str(ROOT))

from utils import (
    ROOT,
    load_cfg,
    write_json,
    artifact_meta,
    build_run_output_path,
    get_run_date,
    canonicalize_filter_spec,
    get_filter_fields_key,
    get_run_id,
    apply_filters,
    ingest,
    ensure_warning_file,
    append_warning,
    get_warning_count,
    get_llm_client,
    start_stage_manifest,
    finalize_stage_manifest,
    build_pipeline_manifest,
    tokens_to_str,
    flat_freq,
    token_doc_freq,
    make_vec,
    add_bin,
    group_key,
    build_project_topic_bridge,
    slugify_group_value,
    load_essay_snippet_lookup,
    quality_report,
    get_effective_support_volume_scale,
    infer_category_family,
    cat_tfidf_slice,
    nmf_one,
    build_input,
    _label_with_retry,
    _norm_group_value,
    _safe_topic_id,
    build_topic_lines,
    build_per_group_prompt,
    _call_with_retry,
    synthesize_one_group,
    strip_json_fences,
    normalize_insight,
    verify_source_topics,
    get_topic_key,
    get_project_ids,
    iter_candidate_insights,
    _parse_topic_id,
    build_full_essay_text,
    build_injected_tokens,
    build_evidence_obj,
    score_insight_confidence,
    _build_rationale_for_row,
    _simulate_tier,
    assign_insight_tier,
    _nano_dedup_decision,
    _topic_list,
    _token_set,
    _screen_pair,
    build_trend_tracker_report_docx,
)

CFG_PATH = ROOT / "params.yaml"
CFG = load_cfg(CFG_PATH)
ct = CFG["tfidf"]
client = get_llm_client()

# Notebook 03 always starts from the enriched project-level parquet produced upstream.
raw_df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")
print(f"Loaded {len(raw_df):,} projects  |  {raw_df.shape[1]} columns")

Loaded 2,244,916 projects  |  46 columns


---
## Parameters

All tunable values live here. Edit this cell; do not edit parameter references in downstream cells.


In [2]:
# ── Runtime setup ───────────────────────────────────────────────────────────
# Keep notebook-level constants close to the top so the rest of the notebook
# reads as pipeline steps instead of repeated CFG lookups.

GROUPBY_FIELD = CFG["analysis"]["group_by"]
GROUP_DESCRIPTION = CFG["analysis"]["group_descriptions"].get(
    GROUPBY_FIELD,
    f"groups defined by '{GROUPBY_FIELD}' in DonorsChoose data",
)
MIN_SHARED = CFG["analysis"]["nmf_min_shared"]
MIN_COVERAGE = CFG["analysis"]["min_coverage"]
BASE_N_COMPONENTS = CFG["nmf"]["n_components"]
CAT_TFIDF_TOP_N = CFG["analysis"]["cat_tfidf_top_n"]
N_REPRESENTATIVE = CFG["llm"]["n_representative_snippets"]
TOP_TERMS_IN_PROMPT = CFG["llm"]["top_terms_in_prompt"]
REVIEW_GROUP = CFG["analysis"]["review_group"]
EXCLUDE_GROUPS = CFG["analysis"]["exclude_groups"]
MAX_WORKERS = CFG["analysis"]["synthesis_max_workers"]
LABELING_MAX_WORKERS = CFG["analysis"].get("labeling_max_workers", MAX_WORKERS)
MODEL_LABELING = CFG["models"]["labeling"]
MODEL_SYNTHESIS = CFG["models"]["synthesis"]
MODEL_VERIFY = CFG["models"]["verify"]
MODEL_DEDUP = CFG["models"]["dedup"]

# Apply the declarative analysis scope once, up front.
FILTER_LOGIC = CFG["analysis"].get("filter_logic", "and")
FILTERS = CFG["analysis"].get("filters", [])
FILTER_SPEC = canonicalize_filter_spec(FILTER_LOGIC, FILTERS)
FILTER_FIELDS_KEY = get_filter_fields_key(FILTERS)

df, filter_summary = apply_filters(raw_df, FILTER_LOGIC, FILTERS)
if filter_summary["no_rows_after_filter"]:
    raise ValueError("No rows remain after applying analysis filters.")

def OUT(subdir, fname):
    return build_run_output_path(
        subdir=subdir,
        fname=fname,
        groupby_field=GROUPBY_FIELD,
        run_date=RUN_DATE,
        run_id=RUN_ID,
    )

# CATEGORY_RULES stays notebook-local for now because it is effectively a
# runtime policy table rather than a pure model parameter block.
CATEGORY_RULES = {
    "broad": {
        "min_group_projects": 50,
        "min_retained_vocab": 80,
        "min_tfidf_nnz": 500,
        "small_slice_cutoff": 175,
        "small_slice_topic_cap": 10,
        "cross_range": (10, 20),
        "per_group_range": (3, 6),
        "cross_section_findings": "2-5",
        "per_group_section_findings": "2-4",
    },
    "medium": {
        "min_group_projects": 35,
        "min_retained_vocab": 60,
        "min_tfidf_nnz": 300,
        "small_slice_cutoff": 125,
        "small_slice_topic_cap": 8,
        "cross_range": (8, 16),
        "per_group_range": (2, 4),
        "cross_section_findings": "1-4",
        "per_group_section_findings": "1-3",
    },
    "sparse": {
        "min_group_projects": 25,
        "min_retained_vocab": 40,
        "min_tfidf_nnz": 200,
        "small_slice_cutoff": 100,
        "small_slice_topic_cap": 6,
        "cross_range": (6, 12),
        "per_group_range": (1, 3),
        "cross_section_findings": "1-3",
        "per_group_section_findings": "1-2",
    },
}

# Adapt slice heuristics to the effective sparsity of the chosen grouping field.
CATEGORY_FAMILY = infer_category_family(GROUPBY_FIELD)
SLICE_RULES = CATEGORY_RULES[CATEGORY_FAMILY].copy()

group_project_counts = (
    df[[GROUPBY_FIELD, "project_id"]]
    .dropna(subset=["project_id"])
    .drop_duplicates()
    .groupby(GROUPBY_FIELD)["project_id"]
    .nunique()
)
median_group_projects = (
    float(group_project_counts.median()) if not group_project_counts.empty else 0.0
)
SLICE_RULES["median_group_projects"] = median_group_projects
SLICE_RULES["small_slice_mode"] = (
    median_group_projects < SLICE_RULES["small_slice_cutoff"]
)

BASE_MIN_GROUP_PROJECTS = CFG["analysis"]["min_group_projects"]
if SLICE_RULES["small_slice_mode"]:
    MIN_GROUP_PROJECTS_EFFECTIVE = min(
        BASE_MIN_GROUP_PROJECTS,
        SLICE_RULES["min_group_projects"],
    )
    CROSS_MIN_INSIGHTS, CROSS_MAX_INSIGHTS = (6, 12)
    PER_GROUP_MIN_INSIGHTS, PER_GROUP_MAX_INSIGHTS = (1, 3)
    CROSS_SECTION_FINDINGS = "1-3"
    PER_GROUP_SECTION_FINDINGS = "1-3"
else:
    MIN_GROUP_PROJECTS_EFFECTIVE = max(
        BASE_MIN_GROUP_PROJECTS,
        SLICE_RULES["min_group_projects"],
    )
    CROSS_MIN_INSIGHTS, CROSS_MAX_INSIGHTS = SLICE_RULES["cross_range"]
    PER_GROUP_MIN_INSIGHTS, PER_GROUP_MAX_INSIGHTS = SLICE_RULES["per_group_range"]
    CROSS_SECTION_FINDINGS = SLICE_RULES["cross_section_findings"]
    PER_GROUP_SECTION_FINDINGS = SLICE_RULES["per_group_section_findings"]

print(
    f"CATEGORY_FAMILY={CATEGORY_FAMILY} | "
    f"small_slice_mode={SLICE_RULES['small_slice_mode']} | "
    f"median_group_projects={median_group_projects:.1f} | "
    f"min_group_projects_effective={MIN_GROUP_PROJECTS_EFFECTIVE}"
)

# Bootstrap run-scoped outputs, warnings, and manifests.
RUN_DATE = get_run_date()
RUN_ID = get_run_id(GROUPBY_FIELD, FILTER_SPEC)

WARNINGS_PATH = OUT("metadata", "warnings_03.jsonl")
ensure_warning_file(WARNINGS_PATH)

FILTER_SPEC_PATH = OUT("metadata", "filter_spec.json")
FILTER_SUMMARY_PATH = OUT("metadata", "filter_summary.json")
COPIED_CONFIG_PATH = OUT("metadata", CFG_PATH.name)

write_json(
    FILTER_SPEC_PATH,
    {
        "schema_version": "v1",
        "run_id": RUN_ID,
        "group_by_field": GROUPBY_FIELD,
        "filter_fields_key": FILTER_FIELDS_KEY,
        "filter_logic": FILTER_LOGIC,
        "filters": FILTERS,
    },
)
write_json(
    FILTER_SUMMARY_PATH,
    {
        "schema_version": "v1",
        "run_id": RUN_ID,
        "group_by_field": GROUPBY_FIELD,
        **filter_summary,
    },
)

shutil.copy2(CFG_PATH, COPIED_CONFIG_PATH)

STAGE_MANIFEST = start_stage_manifest(
    stage_name="03_insights_generation",
    notebook_file="03_insights_generation",
    run_id=RUN_ID,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
)

bins = CFG["analysis"].get("bins", [])
group_cols = [GROUPBY_FIELD, "bin"] if bins else [GROUPBY_FIELD]

print(f"RUN_ID           = {RUN_ID}")
print(f"GROUPBY_FIELD    = {GROUPBY_FIELD!r}")
print(f"FILTER_FIELDS_KEY= {FILTER_FIELDS_KEY}")
print(f"Filtered rows    = {len(df):,} / {len(raw_df):,}")
print(f"Output path      = OUTPUTS/runs/{GROUPBY_FIELD}/{RUN_DATE}/{RUN_ID}/")

CATEGORY_FAMILY=broad | small_slice_mode=False | median_group_projects=1586.0 | min_group_projects_effective=50
RUN_ID           = 20260402_200024_miami_geo_group_438c7b3e
GROUPBY_FIELD    = 'miami_geo_group'
FILTER_FIELDS_KEY= funded_date__miami_geo_group
Filtered rows    = 21,690 / 2,244,916
Output path      = OUTPUTS/runs/miami_geo_group/2026-04-02/20260402_200024_miami_geo_group_438c7b3e/


---
## Step 1 — Build TF-IDF Matrices

One vectorizer fit per n-gram range on the full corpus. Trigrams are persisted
even if unused in Phase 1 analysis — Phase 2+ inherits them without re-extraction.


In [3]:
# Build and cache the reusable TF-IDF matrices once.
# Later cells reuse these objects for QC, category scoring, and debugging.
docs = df["tokens"].apply(tokens_to_str).tolist()

specs = {
    "X_unigram": (1, 1),
    "X_bigram": (2, 2),
    "X_unigram_bigram": (1, 2),
}
if CFG["ngrams"]["max_n"] >= 3:
    specs["X_trigram"] = (3, 3)

matrices, vecs = {}, {}
for name, rng in specs.items():
    vec = make_vec(ct["min_df"], ct["max_df"], rng)
    matrices[name] = vec.fit_transform(docs)
    vecs[name] = vec

    sz, nnz = matrices[name].shape, matrices[name].nnz
    print(f"  {name:20s}: shape={sz}  sparsity={1 - nnz / (sz[0] * sz[1]):.3f}")

# Log top features per matrix to make it easier to spot bad filtering or token drift.
for name, vec in vecs.items():
    top_feats = vec.get_feature_names_out()[:10].tolist()
    print(f"{name:20s} first feats → {top_feats}")

  X_unigram           : shape=(21690, 4859)  sparsity=0.989
  X_bigram            : shape=(21690, 18464)  sparsity=0.999
  X_unigram_bigram    : shape=(21690, 23323)  sparsity=0.997
  X_trigram           : shape=(21690, 3355)  sparsity=0.999
X_unigram            first feats → ['aac', 'abc', 'ability', 'able', 'above', 'absence', 'absent', 'absenteeism', 'absolute', 'absolutely']
X_bigram             first feats → ['aac device', 'ability able', 'ability access', 'ability achieve', 'ability bring', 'ability build', 'ability choose', 'ability communicate', 'ability concentrate', 'ability connect']
X_unigram_bigram     first feats → ['aac', 'aac device', 'abc', 'ability', 'ability able', 'ability access', 'ability achieve', 'ability bring', 'ability build', 'ability choose']
X_trigram            first feats → ['ability concentrate participate', 'ability express themselves', 'ability focus engage', 'ability focus participate', 'ability fully engage', 'ability grow strong', 'ability think cr

---
## Step 2 — Quality Report: Checkpoint 2


In [4]:
qr2 = quality_report(df, label="cp2", matrices=matrices,
                     save_path=OUT("quality", "quality_cp2.json"))


=======================================================  [cp2]
  Projects : 21,690
  Tok/proj : min=16  p50=53  max=99
  Vocab    : 6,814 unique tokens
  X_unigram           : shape=[21690, 4859]  sparsity=0.990
  X_bigram            : shape=[21690, 18464]  sparsity=0.999
  X_unigram_bigram    : shape=[21690, 23323]  sparsity=0.997
  X_trigram           : shape=[21690, 3355]  sparsity=0.999
  Stopwords: FAIL — ['project', 'grade', 'teacher', 'education']



---
## Step 3 — Category TF-IDF

The vectorizer is fit **once** on the full corpus; category slices are scored
by index. This avoids the string-comparison bug (identical token sets across
projects would misclassify rows) and is much faster than refitting per slice.

**Contrast** = token prevalence in this category minus prevalence outside it.

Time bins: defined in `params.yaml` under `analysis.bins`; leave empty for
the full date range.


In [5]:
# ── Category TF-IDF ────────────────────────────────────────────────────────
# Score each eligible group against the rest of the corpus using a shared matrix.

top_n = CAT_TFIDF_TOP_N
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

df_work = add_bin(df, bins) if bins else df.copy()
df_work = df_work.reset_index(drop=True)
all_docs = df_work["tokens"].apply(tokens_to_str).tolist() if bins else docs

vec_cat = make_vec(ct["min_df"], ct["max_df"], tuple(ct["ngram_range"]))
X_full = vec_cat.fit_transform(all_docs)
feat = vec_cat.get_feature_names_out()
idf_vals = vec_cat.idf_

rows = []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj:
        continue

    kd = group_key(keys, group_cols)
    top = cat_tfidf_slice(
        sub.index,
        df_index=df_work.index,
        X_full=X_full,
        feat=feat,
        idf_vals=idf_vals,
        top_n=top_n,
    )
    for col, val in kd.items():
        top.insert(0, col, val)
    rows.append(top)

if not rows:
    raise RuntimeError(
        f"No groups met min_proj={min_proj} threshold — lower min_group_projects in params.yaml"
    )

cat_tfidf_df = pd.concat(rows, ignore_index=True)
cat_tfidf_df.to_csv(OUT("analysis", "category_tfidf.csv"), index=False)

print(f"{len(cat_tfidf_df):,} rows  |  {cat_tfidf_df[group_cols[0]].nunique()} groups")
cat_tfidf_df.head(10)

160 rows  |  8 groups


,miami_geo_group,token,tf,idf,tfidf,prevalence,contrast,project_count
0,"Broward County, FL",bring,0.012582,2.938103,0.036967,0.199739,0.057844,153
1,"Broward County, FL",activity,0.014234,2.510367,0.035733,0.257180,0.037719,197
2,"Broward County, FL",opportunity,0.013027,2.713616,0.035349,0.211488,0.032459,162
3,"Broward County, FL",work,0.017169,2.032824,0.034901,0.360313,0.004502,276
4,"Broward County, FL",fun,0.012033,2.839393,0.034167,0.182768,0.024767,140
5,"Broward County, FL",science,0.010310,3.295282,0.033976,0.134465,0.035010,103
6,"Broward County, FL",goal,0.013047,2.596430,0.033876,0.224543,0.022765,172
7,"Broward County, FL",hand,0.014257,2.344888,0.033432,0.276762,0.016822,212
8,"Broward County, FL",experience,0.013180,2.481964,0.032711,0.245431,0.018944,188
9,"Broward County, FL",engage,0.016136,2.018041,0.032564,0.374674,0.013892,287


---
## Step 4 — NMF Topic Discovery

NMF is fit independently per category so the dominant axis in one category
does not suppress signal in others. Topics are candidates for human review —
not stable theme definitions (that is Phase 2).

The weights DataFrame records which projects load most strongly on each topic;
Step 5 uses it to select representative snippets by NMF weight rather than
randomly.


In [6]:
# ── Groupwise NMF topics ───────────────────────────────────────────────────
# Fit one NMF model per eligible group and keep both topic-level and project-level outputs.

cn = CFG["nmf"]
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

all_topics, all_weights = [], []
df_work = add_bin(df, bins) if bins else df.copy()
df_work = df_work.reset_index(drop=True)

NMF_GROUPS_SKIPPED = []
NMF_GROUPS_FAILED = []

for keys, sub in df_work.groupby(group_cols, observed=True):
    kd = group_key(keys, group_cols)
    group_value = kd[GROUPBY_FIELD]

    # Skip groups that are too small to support stable topic extraction.
    if len(sub) < min_proj:
        NMF_GROUPS_SKIPPED.append(str(group_value))
        continue

    group_docs = sub["tokens"].apply(tokens_to_str).tolist()
    pids = sub["project_id"].tolist()

    try:
        topics, W, nmf_meta = nmf_one(
            group_docs,
            ct_cfg=ct,
            cn_cfg=cn,
            base_n_components=BASE_N_COMPONENTS,
            slice_rules=SLICE_RULES,
        )
    except Exception as e:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF failed for group '{group_value}'",
            context={"group": group_value, "error": str(e)},
        )
        continue

    # Keep skip vs failure separate in the warnings so downstream QA is easier.
    if topics is None or W is None:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF skipped group '{group_value}' because the slice was too thin",
            context={"group": group_value, **(nmf_meta or {})},
        )
        continue

    for col, val in kd.items():
        topics[col] = val
    all_topics.append(topics)

    # Persist the full W ranking so later cells can recover top projects per topic.
    for tid in range(W.shape[1]):
        order = W[:, tid].argsort()[::-1]
        for rank, idx in enumerate(order):
            all_weights.append(
                {
                    **kd,
                    "topic_id": tid,
                    "project_id": pids[idx],
                    "weight": float(W[idx, tid]),
                    "rank": rank,
                }
            )

if not all_topics:
    raise RuntimeError(
        "No groups produced NMF topics — check n_components vs retained vocab, "
        "or lower min_group_projects"
    )

topics_df = pd.concat(all_topics, ignore_index=True)
weights_df = pd.DataFrame(all_weights)

topics_df.to_csv(OUT("analysis", "nmf_topics.csv"), index=False)
weights_df.to_csv(OUT("analysis", "nmf_weights.csv"), index=False)

print(f"{len(topics_df):,} topics across {topics_df[group_cols[0]].nunique()} groups")

# Build the project-topic bridge once so downstream evidence collection can reuse it.
threshold = CFG["analysis"]["topic_assignment_threshold"]
project_topic_bridge_df = build_project_topic_bridge(
    weights_df,
    GROUPBY_FIELD,
    threshold,
)
project_topic_bridge_df.to_csv(OUT("analysis", "project_topic_bridge.csv"), index=False)
print(
    "Bridge: "
    f"{len(project_topic_bridge_df):,} project-topic assignments "
    f"(topic_share >= {threshold})"
)

topics_df.head(6)

/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 400 reached. Increase it to improve convergence.
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 400 reached. Increase it to improve convergence.
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 400 reached. Increase it to improve convergence.
  warnings.warn(


160 topics across 8 groups
Bridge: 17,936 project-topic assignments (topic_share >= 0.3)


,topic_id,top_terms,top_weights,miami_geo_group
0,0,"[letter, literacy, word, sound, interactive, i...","[0.8256620664818485, 0.7894686070443306, 0.661...","Broward County, FL"
1,1,"[focus, stay, challenge, succeed, environment,...","[0.5010442001030759, 0.43458181240228333, 0.43...","Broward County, FL"
2,2,"[able, some, teach, work, test, level, one, st...","[0.4208331739996819, 0.3373969738906314, 0.324...","Broward County, FL"
3,3,"[organize, keep, folder, paper, organization, ...","[0.5799744886014903, 0.4732027287735601, 0.421...","Broward County, FL"
4,4,"[solve, problem, problem solve, hand, explore,...","[0.5283520301251665, 0.520273493182175, 0.4572...","Broward County, FL"
5,5,"[seat, flexible, comfortable, option, flexible...","[0.5311794586495191, 0.48446952119496156, 0.39...","Broward County, FL"


In [7]:
# ── Cross-group universal themes ───────────────────────────────────────────
# Identify term bundles that recur across groups after topic extraction.

if REVIEW_GROUP is not None:
    _eligible = topics_df[group_cols[0]].unique().tolist()
    if REVIEW_GROUP not in _eligible:
        raise ValueError(f"REVIEW_GROUP={REVIEW_GROUP!r} not found. Eligible: {_eligible}")
else:
    REVIEW_GROUP = df_work.groupby(GROUPBY_FIELD).size().idxmax()

records = [(r[group_cols[0]], frozenset(r.top_terms)) for _, r in topics_df.iterrows()]

# theme_cats maps a shared term bundle to the set of groups where it recurs.
theme_cats = defaultdict(set)
for i, (ci, si) in enumerate(records):
    for cj, sj in records[i + 1:]:
        if ci == cj:
            continue
        shared = si & sj
        if len(shared) >= MIN_SHARED:
            key = frozenset(shared)
            theme_cats[key] |= {ci, cj}

rows = sorted(theme_cats.items(), key=lambda x: -len(x[1]))
seen = []
deduped = []
for terms, cats in rows:
    if len(cats) < MIN_COVERAGE:
        continue
    if not any(terms <= prior for prior in seen):
        deduped.append(
            {
                "theme": ", ".join(sorted(terms)[:5]),
                "n_groups": len(cats),
                "categories": sorted(cats),
            }
        )
        seen.append(terms)

cross_group_df = pd.DataFrame(deduped).reset_index(drop=True)
cross_group_df.to_csv(OUT("analysis", "cross_group_themes.csv"), index=False)
cross_group_df

,theme,n_groups,categories
0,"interest, read, reader",7,"[Broward County, FL, Houston, TX, Las Vegas, N..."
1,"book, interest, library, love, read",6,"[Houston, TX, Las Vegas, NV, Los Angeles, CA, ..."
2,"concept, hand, science, world",6,"[Houston, TX, Las Vegas, NV, Los Angeles, CA, ..."


---
## Step 5 — LLM Topic Labeling

One API call per topic on compressed input only — never raw essay text at scale.
Representative snippets are chosen by NMF weight (not random).

Prompt field limits (`top_unigrams`, `top_bigrams`, `top_nmf_terms`) are applied
as item counts. Parse failures are stored with enough metadata to debug later.


In [8]:
# ── LLM topic labeling ─────────────────────────────────────────────────────
# Convert raw NMF topics into human-readable labels/descriptions with retry logic.

SYSTEM = (
    "You are an NLP analyst reviewing NMF topic clusters from DonorsChoose teacher "
    "essays. Respond ONLY with a JSON object, no preamble, no markdown fences.\n\n"
    "Token naming conventions you may see in topic terms:\n"
    "  __framing_[name]__ = a rhetorical framing signal injected during preprocessing "
    "(e.g. __framing_barrier_removal__ means essays removing obstacles to participation)\n"
    "  __cat_[name]__ = a subject matter category token (e.g. __cat_genetics__)\n"
    "  __sub_[name]__ = a subcategory token\n"
    "These are meaningful signals — incorporate them into your label and description.\n\n"
    "Labeling rules:\n"
    "- Prefer the most specific defensible label.\n"
    "- Preserve concrete signals such as named programs, vendors, pedagogies, student "
    "populations, classroom use cases, or rhetorical framing when present.\n"
    "- Do not collapse topics into broad umbrella labels like technology, literacy, "
    "engagement, classroom supplies, or social-emotional learning if the evidence "
    "supports a narrower interpretation.\n"
    "- When framing tokens are present, treat them as first-class evidence about how "
    "teachers are positioning the request, not just background metadata.\n"
    "- If a topic is mixed, say what subthemes are colliding rather than forcing an "
    "overly neat label.\n\n"
    "coherence_flag definitions:\n"
    "  coherent   — terms and representative snippets point clearly to one theme, "
    "population, or use case.\n"
    "  mixed      — two or more distinguishable subthemes are colliding in this topic; "
    "describe what they are in notes.\n"
    "  redundant  — this topic's terms and snippets substantially duplicate another "
    "topic in the same group, not merely overlap in subject area.\n"
    "  unclear    — terms are too scattered or generic to support a defensible label; "
    "the topic does not resolve to a coherent signal."
)

PROMPT = (
    f"Group ({GROUPBY_FIELD}): {{group_value}}{{bin_line}}\n"
    "Topic {topic_id}\n"
    "Top unigrams : {unigrams}\n"
    "Top bigrams  : {bigrams}\n"
    "Top NMF terms: {nmf_terms}\n"
    "Representative project tokens:\n"
    "{snippets}\n\n"
    "Instructions:\n"
    "- proposed_label should be concrete and specific, ideally naming the request or "
    "intervention, the population or context, and the framing when supported.\n"
    "- description should explain what makes this topic distinct from nearby generic topics.\n"
    "- If the topic appears broad, identify the narrower mechanism, use case, or population "
    "if the evidence supports it.\n"
    "- If the evidence is genuinely mixed, preserve that ambiguity instead of inventing a "
    "falsely precise label.\n\n"
    f"Return a JSON object with exactly these keys:\n"
    f"  {GROUPBY_FIELD}, topic_id, proposed_label, description, coherence_flag, notes\n"
    'coherence_flag must be one of: "coherent", "mixed", "redundant", "unclear" '
    "(definitions provided in system instructions)"
)

# Build lightweight snippet text from the top projects attached to each topic.
pid_text = df.set_index("project_id")["tokens"].apply(lambda t: " ".join(t[:40]))
topic_inputs = [
    build_input(
        t,
        weights_df=weights_df,
        pid_text=pid_text,
        groupby_field=GROUPBY_FIELD,
        n_representative=N_REPRESENTATIVE,
        top_terms_in_prompt=TOP_TERMS_IN_PROMPT,
    )
    for _, t in topics_df.iterrows()
]

topic_order = {
    (_norm_group_value(inp["group_value"]), int(inp["topic_id"])): i
    for i, inp in enumerate(topic_inputs)
}

results = []
with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _label_with_retry,
            inp,
            client=client,
            model_labeling=MODEL_LABELING,
            system_prompt=SYSTEM,
            user_prompt_template=PROMPT,
            groupby_field=GROUPBY_FIELD,
            warnings_path=WARNINGS_PATH,
        ): inp
        for inp in topic_inputs
    }
    for future in as_completed(futures):
        inp = futures[future]
        obj = future.result()
        results.append(obj)
        print(f"  {inp['group_value']} / topic {inp['topic_id']} → {obj.get('proposed_label', '?')}")

# Re-sort to the original topic order so downstream joins stay stable.
bad_sort_keys = []
for r in results:
    norm_key = (
        _norm_group_value(r.get(GROUPBY_FIELD, "")),
        _safe_topic_id(r.get("topic_id", -1)),
    )
    if norm_key not in topic_order:
        bad_sort_keys.append({"group": r.get(GROUPBY_FIELD, ""), "topic_id": r.get("topic_id", "")})

if bad_sort_keys:
    raise ValueError(
        f"LABELING_SORT_KEY_MISMATCH: {len(bad_sort_keys)} label result(s) did not match the input topic order. "
        f"Examples: {bad_sort_keys[:5]}"
    )

results = sorted(
    results,
    key=lambda r: topic_order[
        (_norm_group_value(r.get(GROUPBY_FIELD, "")), _safe_topic_id(r.get("topic_id", -1)))
    ],
)

with open(OUT("analysis", "llm_topic_labels.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)

print(f"\n{len(results)} labels saved")

  Broward County, FL / topic 5 → Flexible seating for student choice/autonomy (__framing_student_choice_autonomy__, portable classroom furniture)
  Broward County, FL / topic 6 → Math resources (e.g., calculators) framed as humble gratitude and barrier removal for equitable access
  Broward County, FL / topic 1 → __framing_[challenge_to_succeed]__ focus/success routines for resilient and ESOL/Kindergarten learners using “ready-to-focus” classroom environment supports
  Broward County, FL / topic 4 → __cat_stem__/__cat_science__ hands-on STEAM makerspace & coding kits (Botley, Makey Makey, micro:bit) for problem-solving, exploration, and teamwork
  Broward County, FL / topic 7 → Art instruction framed around culturally diverse self-portrait/multicultural projects with social-emotional calming (__framing_social_emotional__)
  Broward County, FL / topic 3 → Organizational storage systems for student work (folders/binders/grader folders) framed as consumables replenishment, triage/repair, 

In [9]:
# ── Label post-processing ──────────────────────────────────────────────────
# Keep only parseable labeling results and validate them against source topics.

parse_errors = [r for r in results if r.get("parse_error")]
if parse_errors:
    print(f"WARNING: {len(parse_errors)} topics failed JSON parse — excluded from synthesis:")
    for e in parse_errors:
        print(f"  {e.get(GROUPBY_FIELD, '?')} / topic {e.get('topic_id', '?')}")

labels_df = pd.DataFrame([r for r in results if not r.get("parse_error")])

assert GROUPBY_FIELD in labels_df.columns, (
    f"labels_df missing '{GROUPBY_FIELD}' — re-run labeling for this groupby field"
)

# Source-of-truth allowed groups come from the source topic table, not model output.
ALLOWED_GROUP_VALUES = [
    str(g)
    for g in topics_df[GROUPBY_FIELD].dropna().drop_duplicates().tolist()
]

invalid_groups = sorted(
    set(labels_df[GROUPBY_FIELD].astype(str).unique()) - set(ALLOWED_GROUP_VALUES)
)
if invalid_groups:
    raise ValueError(
        f"Invalid {GROUPBY_FIELD} values returned during labeling: {invalid_groups}"
    )

# Normalize types after validation so later joins are stable.
labels_df[GROUPBY_FIELD] = labels_df[GROUPBY_FIELD].astype(str)
labels_df["topic_id"] = labels_df["topic_id"].astype(int)

labeled_groups = set(labels_df[GROUPBY_FIELD].unique()) if not labels_df.empty else set()
LABELING_FAILED_GROUPS = sorted(set(ALLOWED_GROUP_VALUES) - labeled_groups)

labels_df.groupby("coherence_flag").size().rename("count").to_frame()
labels_df[[GROUPBY_FIELD, "topic_id", "proposed_label", "coherence_flag", "description"]]

,miami_geo_group,topic_id,proposed_label,coherence_flag,description
0,"Broward County, FL",0,__framing_milestone_culmination_framing__ kind...,coherent,This topic centers on early childhood (kinderg...
1,"Broward County, FL",1,__framing_[challenge_to_succeed]__ focus/succe...,coherent,This topic clusters around building a “ready t...
2,"Broward County, FL",2,Instructional support for assessment/competiti...,mixed,Projects center on helping students participat...
3,"Broward County, FL",3,Organizational storage systems for student wor...,coherent,This topic clusters around classroom organizat...
4,"Broward County, FL",4,__cat_stem__/__cat_science__ hands-on STEAM ma...,coherent,This topic clusters teacher requests for hands...
...,...,...,...,...,...
155,"Tampa, FL",15,__cat_special_needs__/__framing_readiness_to_l...,coherent,This topic centers on using structured play (g...
156,"Tampa, FL",16,Classroom “safe/cozy” sensory reading & meetin...,coherent,"This topic centers on designing a comforting, ..."
157,"Tampa, FL",17,Behaviorist reinforcement for academic complia...,coherent,This topic centers on motivating and shaping s...
158,"Tampa, FL",18,Interactive digital literacy and classroom out...,coherent,This topic centers on using interactive/techno...


In [10]:
# ── SYNTHESIS — cross-group + per-group loops ─────────────────────────────
# First synthesize the whole landscape, then synthesize each group in parallel.

SYNTHESIS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You write precise, insight-driven briefings for internal strategy audiences. "
    "You do not pad your answers or state the obvious.\n\n"
    "Some topic labels may reference rhetorical framing signals (e.g. 'barrier removal "
    "framing', 'chronic scarcity language') — these come from injected tokens that "
    "categorize how teachers write, not just what they request. Treat these as "
    "meaningful signals about teacher rhetorical strategy.\n\n"
    "Prioritize nuance over novelty. Prefer fine-grained distinctions, concrete "
    "mechanisms, specific populations, and framing differences over broad thematic "
    "summaries.\n\n"
    "Grounding: when writing Framing Differences findings, the contrast must be "
    "traceable to explicit language in the topic label or description — not inferred "
    "from the topic's general subject matter or implied by category context. If you "
    "cannot quote or closely paraphrase specific label language to establish both sides "
    "of a contrast, do not make the finding."
)

topic_lines = build_topic_lines(labels_df, GROUPBY_FIELD)

SYNTHESIS_PROMPT_CROSS_GROUP = f"""
Below is a list of NMF topics discovered from teacher project request essays on DonorsChoose,
grouped by {GROUPBY_FIELD} ({GROUP_DESCRIPTION}).
Each topic represents a cluster of essays with similar language, framing, and request patterns.
{topic_lines}

Your task: identify the most specific, decision-useful patterns in this topic landscape.

Rules:
- Every finding must be grounded in specific topic labels, descriptions, or coherence flags
  from the list above.
- Do not introduce concepts, themes, or absences that cannot be traced to an actual topic.
- If you cannot name the specific group and topic, do not make the finding.
- Prefer fine-grained distinctions over broad summaries.
- Skip anything predictable given the group name.
- Skip the free/reduced lunch poverty framing — it appears everywhere and is already known.
- Skip generic engagement/motivation language unless the topic evidence makes it unusually specific.
- If fewer than two strong findings exist for a section, write one or write none. Do not pad.

Focus on:
1. RECURRING INTERNAL SPLITS — cases where the same subtheme distinction recurs independently
   inside multiple groups. Only flag a split if it appears in at least two groups with
   meaningfully similar internal logic. Name the groups and specific topics each time.

2. CROSS-GROUP SIGNATURES — a specific program, vendor, pedagogy, student need, or rhetorical
   frame that shows up as a recognizable topic in multiple groups. Name every group it appears
   in and explain why the pattern is meaningful.

3. FRAMING DIFFERENCES — cases where similar underlying needs are presented through meaningfully
   different rhetorical frames, populations, or classroom contexts. Explain what changes across
   groups and why that difference matters.

4. NOTABLE ABSENCES — a topic you would expect given the internal logic of a group's other
   topics, but that is missing. To qualify, name at least two specific topics already present
   in the same group that make the gap conspicuous from within. Do not reason from other groups.

5. REQUEST TRIGGERS AND PITCH STRATEGIES — patterns in why teachers say they are requesting and
   how they frame those reasons for a donor audience. Only use explicit topic evidence.

Formatting: use plain numbered section headers (e.g. '1. SUBTHEME DISTINCTIONS').
Do not use markdown headers (##). Do not use bullet points.

Format as five labeled sections. Under each, write {CROSS_SECTION_FINDINGS} findings as short paragraphs.
Be specific — name the group and topic label every time. Do not use bullet points.
""".strip()

PER_GROUP_INSTRUCTIONS = """
Your task: identify the most specific, decision-useful patterns within this group.

Rules:
- Every finding must be grounded in specific topic labels, descriptions, or coherence flags
  from the list above.
- Do not introduce concepts, themes, or absences that cannot be traced to an actual topic.
- If you cannot name the specific topic, do not make the finding.
- Prefer fine-grained distinctions over broad summaries.
- Skip anything predictable given the group name.
- Skip the free/reduced lunch poverty framing — it appears everywhere and is already known.
- Skip generic engagement/motivation language unless the topic evidence makes it unusually specific.
- If fewer than two strong findings exist for a section, write one or write none. Do not pad.

Focus on:
1. SUBTHEME DISTINCTIONS — distinct needs, populations, pedagogies, use cases, or framing
   modes within this group.

2. INTERNAL FRAMING DIFFERENCES — cases within this group where similar underlying needs are
   presented through meaningfully different rhetorical frames, populations, or classroom
   contexts.

3. NOTABLE ABSENCES — a topic you would expect given the internal logic of this group's other
   topics, but that is missing.

4. REQUEST TRIGGERS AND PITCH STRATEGIES — patterns in why teachers say they are requesting
   within this group and how they frame that reason for a donor audience.

Formatting: use plain numbered section headers (e.g. '1. SUBTHEME DISTINCTIONS').
Do not use markdown headers (##). Do not use bullet points.

Format as four labeled sections. Under each, write {PER_GROUP_SECTION_FINDINGS} findings as short paragraphs.
Be specific — name the topic label every time. Do not use bullet points.
""".strip()

synthesis_cross = _call_with_retry(
    SYNTHESIS_PROMPT_CROSS_GROUP,
    client=client,
    model_name=MODEL_SYNTHESIS,
    system_prompt=SYNTHESIS_SYSTEM,
)
if synthesis_cross is None:
    append_warning(
        WARNINGS_PATH,
        "03_insights_generation",
        "SYNTHESIS_CROSS_GROUP_FAILED",
        "Cross-group synthesis failed",
        context={},
    )
    raise RuntimeError("Cross-group synthesis failed; stopping before downstream cells.")

with open(OUT("analysis", "llm_synthesis_cross_group.txt"), "w", encoding="utf-8") as f:
    f.write(synthesis_cross)

groups = [
    g for g in ALLOWED_GROUP_VALUES
    if g not in EXCLUDE_GROUPS and g in set(labels_df[GROUPBY_FIELD].unique())
]

per_group_results = {}
SYNTHESIS_FAILED_GROUPS = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            synthesize_one_group,
            g,
            labels_df=labels_df,
            groupby_field=GROUPBY_FIELD,
            group_description=GROUP_DESCRIPTION,
            per_group_instructions=PER_GROUP_INSTRUCTIONS,
            client=client,
            model_name=MODEL_SYNTHESIS,
            system_prompt=SYNTHESIS_SYSTEM,
            warnings_path=WARNINGS_PATH,
            outpath_func=OUT,
        ): g
        for g in groups
    }
    for future in as_completed(futures):
        submitted_group = futures[future]
        try:
            group, result = future.result()
        except Exception as e:
            SYNTHESIS_FAILED_GROUPS.append(str(submitted_group))
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "SYNTHESIS_GROUP_FAILED",
                f"Synthesis failed for group '{submitted_group}'",
                context={"group": submitted_group, "error": str(e)},
            )
            print(f"FAILED: {submitted_group} | error: {e}")
            continue

        if result is not None:
            per_group_results[group] = result
            print(f"Done: {group}")
        else:
            SYNTHESIS_FAILED_GROUPS.append(str(group))
            print(f"FAILED: {group}")

# Preserve the original group order before handing results downstream.
per_group_results = {g: per_group_results[g] for g in groups if g in per_group_results}

if not per_group_results:
    raise RuntimeError("No per-group synthesis results were produced; stopping before downstream cells.")

Done: San Diego, CA
Done: Houston, TX
Done: Tampa, FL
Done: Orlando, FL
Done: Broward County, FL
Done: Las Vegas, NV
Done: Miami, FL
Done: Los Angeles, CA


In [11]:
# ── EXTERNAL-FACING INSIGHTS — structured JSON call ───────────────────────
# Ask the model for a single JSON object, then normalize and validate it.

OUTPUT_GROUP_KEY = "by_group"
REQUIRED_GROUP_VALUES = list(per_group_results.keys())

if not REQUIRED_GROUP_VALUES:
    raise RuntimeError("No synthesized group results are available; cannot build external-facing insights.")

synthesis_parts = []
if synthesis_cross:
    synthesis_parts.append(f"=== CROSS-GROUP ANALYSIS ===\n{synthesis_cross}")
for group_value, text in per_group_results.items():
    if text:
        synthesis_parts.append(f"=== {group_value} ===\n{text}")
synthesis_input = "\n\n".join(synthesis_parts)

INSIGHTS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You write precise, decision-facing briefings for external audiences: "
    "foundations, corporate partners, policymakers, and major donors. "
    "You return ONLY valid JSON. No preamble, no explanation, no markdown fences."
)

example_source_topics = []
example_rows = (
    labels_df[[GROUPBY_FIELD, "topic_id"]]
    .drop_duplicates()
    .head(3)
    .to_dict("records")
)

for row in example_rows:
    example_source_topics.append(f"{row[GROUPBY_FIELD]}|{int(row['topic_id'])}")

if not example_source_topics:
    for i, gv in enumerate(REQUIRED_GROUP_VALUES[:2], start=1):
        example_source_topics.append(f"{gv}|{i}")

example_source_topics_json = json.dumps(example_source_topics, ensure_ascii=False)

INSIGHTS_PROMPT = f"""
You are given a comprehensive structured analysis of classroom project request patterns including:
within-group subthemes, cross-group signatures, framing differences, notable absences,
and request triggers and pitch strategies.

The analysis is grouped by the field "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

{synthesis_input}

Your task: produce a decision-facing insights document for external stakeholders —
executives, foundation leaders, state legislators, and major donors — who are
intelligent but not close to classroom realities. These readers should finish each
insight thinking: "I never would have known that, and the fact that this organization
can see it means I should take their priorities seriously."

SELECTION CRITERIA:
- Each insight must surface something genuinely non-obvious about what classrooms need,
  reframe how a thoughtful outsider should interpret teacher demand, and have a clear
  implication for where attention or resources should go.
- Prioritize insights that reveal hidden scale, misallocated attention, or a shift in
  how teachers are operating that external stakeholders would not expect.
- Avoid methodological commentary, insights that only matter to data practitioners,
  and observations that merely restate what the group label already suggests.

COUNT + COVERAGE REQUIREMENTS:
- Return between {CROSS_MIN_INSIGHTS} and {CROSS_MAX_INSIGHTS} cross-group insights in "key_insights". Produce as many as
  the evidence strongly supports — do not pad to reach a target and do not stop early
  if strong findings remain.
- "{OUTPUT_GROUP_KEY}" must include every group value in this exact list:
  {json.dumps(REQUIRED_GROUP_VALUES, ensure_ascii=False)}
- Return {PER_GROUP_MIN_INSIGHTS}-{PER_GROUP_MAX_INSIGHTS} strongest defensible insights for each group value above.
- Do not omit group values silently.
- Prefer slightly weaker but still defensible coverage over omission.

INSIGHT STRUCTURE — for every insight use exactly these fields:
- title: a crisp, declarative headline that makes the finding feel like a reveal,
  not a description.
- what_seeing: 2-4 sentences describing the observed pattern in plain, concrete language.
  Write about what teachers are actually doing or asking for — not about the analysis.
- why_easy_to_miss: 1-2 sentences explaining why this pattern is invisible to someone
  looking at surface-level groups or conventional reporting.
- source_topics: a list of the strongest grounding topics for the insight.
  Use only topics that directly support the claim.
  Usually 2-4 topics is enough, but do not force a fixed count.
  Prefer fewer well-matched topics over padding with weak ones.
  Required format:
    - each item must be a string
    - each string must be exactly "<group_value>|<topic_id>"
    - group_value must exactly match one of the group values shown in the analysis
    - topic_id must be the integer topic number for that group
  Do not use labels, category names, prose, or invented group names in place of group_value.
  Example for this run: {example_source_topics_json}
- Do not include a "section" field inside each insight.
  Section is implied by where the insight appears in the JSON.

STYLE:
- Direct, confident, and human — not academic, not corporate.
- No mention of "topics", "model", "NMF", "pipeline", "cluster", or analytical machinery.
- No hedging phrases like "may suggest" or "could indicate".
- No filler.
- Do not repeat the same implication across insights.

VALIDITY RULES:
- Return valid JSON only.
- The response is incomplete if any required group value is missing from "{OUTPUT_GROUP_KEY}".
- Do not return markdown fences or explanatory text.

Return a JSON object with exactly this structure:
{{
  "key_insights": [/* {CROSS_MIN_INSIGHTS}-{CROSS_MAX_INSIGHTS} insight objects */],
  "{OUTPUT_GROUP_KEY}": {{
    "<group value>": [/* {PER_GROUP_MIN_INSIGHTS}-{PER_GROUP_MAX_INSIGHTS} insight objects */],
    ...
  }}
}}
""".strip()

resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    messages=[
        {"role": "system", "content": INSIGHTS_SYSTEM},
        {"role": "user", "content": INSIGHTS_PROMPT},
    ],
    response_format={"type": "json_object"},
)

raw_json = strip_json_fences(resp.choices[0].message.content)
insights_data = json.loads(raw_json)

if not isinstance(insights_data, dict):
    raise ValueError("Top-level insights response is not a JSON object.")

raw_key = insights_data.get("key_insights", [])
raw_by_group = insights_data.get(OUTPUT_GROUP_KEY, {})

if not isinstance(raw_key, list):
    raise ValueError("key_insights is not a list.")
if not isinstance(raw_by_group, dict):
    raise ValueError(f"{OUTPUT_GROUP_KEY} is not an object.")

normalized = {
    "key_insights": [
        normalize_insight(i, required_group_values=REQUIRED_GROUP_VALUES)
        for i in raw_key
    ],
    OUTPUT_GROUP_KEY: {},
}

missing_group_values = [g for g in REQUIRED_GROUP_VALUES if g not in raw_by_group]
if missing_group_values:
    raise ValueError(f"Model omitted required group values: {missing_group_values}")

extra_group_values = [g for g in raw_by_group.keys() if g not in REQUIRED_GROUP_VALUES]
if extra_group_values:
    print(f"WARNING: extra group values returned and ignored: {extra_group_values}")

for group_value in REQUIRED_GROUP_VALUES:
    items = raw_by_group.get(group_value, [])
    if not isinstance(items, list):
        raise ValueError(f"{OUTPUT_GROUP_KEY}['{group_value}'] is not a list.")
    normalized[OUTPUT_GROUP_KEY][group_value] = [
        normalize_insight(i, required_group_values=REQUIRED_GROUP_VALUES)
        for i in items
    ]

insights_data = normalized

if not (CROSS_MIN_INSIGHTS <= len(insights_data["key_insights"]) <= CROSS_MAX_INSIGHTS):
    raise ValueError(
        f"Expected between {CROSS_MIN_INSIGHTS} and {CROSS_MAX_INSIGHTS} key insights, "
        f"got {len(insights_data['key_insights'])}"
    )

bad_group_counts = {
    group_value: len(items)
    for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    if not (PER_GROUP_MIN_INSIGHTS <= len(items) <= PER_GROUP_MAX_INSIGHTS)
}
if bad_group_counts:
    raise ValueError(
        f"Expected between {PER_GROUP_MIN_INSIGHTS} and {PER_GROUP_MAX_INSIGHTS} insights per group value, "
        f"got: {bad_group_counts}"
    )

# Mid-cell write required by spec: post-normalize_insight(), pre-verification.
write_json(OUT("insights", "insights_candidates.json"), insights_data)

PosixPath('/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/miami_geo_group/2026-04-02/20260402_200024_miami_geo_group_438c7b3e/insights/insights_candidates.json')

In [12]:
# Quick structural sanity check before topic verification and evidence building.
total_source_topics = (
    sum(len(i.get("source_topics", [])) for i in insights_data["key_insights"])
    + sum(
        len(i.get("source_topics", []))
        for items in insights_data[OUTPUT_GROUP_KEY].values()
        for i in items
    )
)

print(f"Key insights:  {len(insights_data.get('key_insights', []))}")
print(f"Group values:  {list(insights_data.get(OUTPUT_GROUP_KEY, {}).keys())}")
print(f"Total source_topics across all insights: {total_source_topics}")

Key insights:  13
Group values:  ['Broward County, FL', 'Houston, TX', 'Las Vegas, NV', 'Los Angeles, CA', 'Miami, FL', 'Orlando, FL', 'San Diego, CA', 'Tampa, FL']
Total source_topics across all insights: 178


In [13]:
# ── Topic verification ─────────────────────────────────────────────────────
# Re-check each synthesized source topic against the labeled-topic inventory.

VERIFY_SYSTEM = (
    "You are checking whether topic labels correctly support a given insight. "
    "Be strict. If the topic is only loosely related, drop it. "
    "Return only valid JSON."
)

for key in ["key_insights"]:
    insights_data[key] = [
        verify_source_topics(
            i,
            labels_df=labels_df,
            groupby_field=GROUPBY_FIELD,
            required_group_values=REQUIRED_GROUP_VALUES,
            client=client,
            model_verify=MODEL_VERIFY,
            system_prompt=VERIFY_SYSTEM,
            warnings_path=WARNINGS_PATH,
        )
        for i in insights_data.get(key, [])
    ]

for cat in insights_data.get(OUTPUT_GROUP_KEY, {}):
    insights_data[OUTPUT_GROUP_KEY][cat] = [
        verify_source_topics(
            i,
            labels_df=labels_df,
            groupby_field=GROUPBY_FIELD,
            required_group_values=REQUIRED_GROUP_VALUES,
            client=client,
            model_verify=MODEL_VERIFY,
            system_prompt=VERIFY_SYSTEM,
            warnings_path=WARNINGS_PATH,
        )
        for i in insights_data[OUTPUT_GROUP_KEY][cat]
    ]

print("Verification complete.")

Adjusted source_topics: Literacy demand has split into two different markets: decoding tools and reading | 8 -> 3
Adjusted source_topics: Device access is no longer a single category; audio access has become its own in | 7 -> 6
Adjusted source_topics: Gift cards have become a classroom procurement system, not a side request. | 5 -> 4
Adjusted source_topics: Dry-erase materials are being used as a low-cost feedback engine, especially in  | 5 -> 3
Adjusted source_topics: Teachers increasingly pitch invisible operating costs, not enrichment, because c | 6 -> 5
Adjusted source_topics: Basic-needs requests are becoming instructionally targeted, not generic welfare  | 6 -> 5
Adjusted source_topics: Writing is the quiet gap in teacher demand, even where reading support is highly | 6 -> 1
Adjusted source_topics: Broward teachers have built three separate literacy strategies, not one reading  | 3 -> 0
Adjusted source_topics: Broward teachers make modest purchases fundable by naming the exact in

In [14]:
# ── EVIDENCE PACK + CONFIDENCE SCORING ─────────────────────────────────────
# Convert verified source topics into project-level evidence, then score confidence.

assert "project_topic_bridge_df" in dir() and not project_topic_bridge_df.empty, (
    "project_topic_bridge_df missing — ensure the bridge-build cell ran successfully"
)

# BRIDGE_LOOKUP lets us jump from a verified topic to its supporting projects cheaply.
BRIDGE_LOOKUP = {
    topic_key: grp[["project_id", "weight", "topic_share"]].copy()
    for topic_key, grp in project_topic_bridge_df.groupby("topic_key", observed=True)
}

candidate_rows = list(iter_candidate_insights(insights_data, OUTPUT_GROUP_KEY))

topic_rows_cache_global = {}
all_needed_pids = []
for row in candidate_rows:
    _, topic_rows_cache = get_project_ids(
        row["insight"].get("source_topics", []),
        BRIDGE_LOOKUP,
        groupby_field=GROUPBY_FIELD,
        max_ids=None,
    )
    for topic_key, topic_rows in topic_rows_cache.items():
        if topic_key not in topic_rows_cache_global:
            topic_rows_cache_global[topic_key] = topic_rows
        all_needed_pids.extend(topic_rows["project_id"].tolist())

all_needed_pids = list(dict.fromkeys(all_needed_pids))

ESSAY_LOOKUP_PATH = ROOT / "OUTPUTS/prepared/02_project_essay_lookup.parquet"
if not ESSAY_LOOKUP_PATH.exists():
    raise FileNotFoundError(
        f"Essay lookup parquet not found: {ESSAY_LOOKUP_PATH}. Run Notebook 01 first."
    )

# Pull only the essay rows we actually need for accepted/rejected evidence exports.
essay_lookup_df = pd.read_parquet(
    ESSAY_LOOKUP_PATH,
    columns=["project_id", "essay_text"],
)
essay_lookup = (
    essay_lookup_df[essay_lookup_df["project_id"].isin(all_needed_pids)]
    .drop_duplicates(subset=["project_id"], keep="first")
    .set_index("project_id")["essay_text"]
    .to_dict()
)

token_lookup = df.set_index("project_id")["tokens"].to_dict()

# label_index is the lookup table used when converting source topics into evidence.
label_index = {}
for _, row in labels_df.iterrows():
    try:
        tid = _parse_topic_id(row["topic_id"])
        label_index[(str(row[GROUPBY_FIELD]), tid)] = row
    except (ValueError, IndexError):
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "LABEL_INDEX_PARSE_FAILURE",
            f"Could not parse topic_id '{row['topic_id']}' for group '{row[GROUPBY_FIELD]}'",
            context={"group": str(row[GROUPBY_FIELD]), "topic_id": str(row["topic_id"])},
        )

candidate_evidence_rows = [
    build_evidence_obj(
        row,
        groupby_field=GROUPBY_FIELD,
        run_id=RUN_ID,
        n_representative=N_REPRESENTATIVE,
        topic_rows_cache_global=topic_rows_cache_global,
        label_index=label_index,
    )
    for row in candidate_rows
]

CONFIDENCE_CFG = dict(CFG["analysis"]["confidence"])

support_volume_scale_meta = get_effective_support_volume_scale(
    df=df,
    groupby_field=GROUPBY_FIELD,
    confidence_cfg=CONFIDENCE_CFG,
)
CONFIDENCE_CFG["support_volume_scale_effective"] = support_volume_scale_meta["effective_scale"]

print(
    f"Effective support_volume_scale_effective: {CONFIDENCE_CFG['support_volume_scale_effective']} "
    f"(multiplier={support_volume_scale_meta['scale_multiplier']}, "
    f"median_group_size={support_volume_scale_meta['median_group_size']:.1f})"
)

TOPIC_QUALITY_MAP = {
    "coherent": 1.0,
    "mixed": 0.6,
    "redundant": 0.2,
    "unclear": 0.0,
    "unknown": 0.0,
}

CONFIDENCE_RATIONALE_SYSTEM = (
    "You explain confidence scores for analytical insights. Return only one short sentence."
)

conf_rows = []
for ev in candidate_evidence_rows:
    conf = score_insight_confidence(
        ev,
        cfg=CONFIDENCE_CFG,
        topic_quality_map=TOPIC_QUALITY_MAP,
    )
    conf["insight_id"] = ev["insight_id"]
    conf["title"] = ev["title"]
    conf["confidence_version"] = "v1"
    conf_rows.append(conf)

# Generate short rationale sentences in parallel; these are explanatory only.
rationale_map = {}
with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _build_rationale_for_row,
            row,
            client=client,
            model_name=MODEL_LABELING,
            system_prompt=CONFIDENCE_RATIONALE_SYSTEM,
        ): row["insight_id"]
        for row in conf_rows
    }
    for future in as_completed(futures):
        try:
            insight_id, rationale = future.result()
            rationale_map[insight_id] = rationale
        except Exception as e:
            insight_id = futures[future]
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "RATIONALE_API_FAILURE",
                f"Confidence rationale failed for '{insight_id}'",
                context={"insight_id": insight_id, "error": str(e)},
            )
            rationale_map[insight_id] = None

for row in conf_rows:
    row["confidence_rationale"] = rationale_map.get(row["insight_id"])

conf_df = pd.DataFrame(conf_rows)
conf_df.to_csv(OUT("insights", "insights_confidence.csv"), index=False)
print(f"Scored {len(conf_df):,} candidate insights")

# ── STRICT CHANGE VERIFICATION ────────────────────────────────────────────
# Confirms that the refactor preserved the same structural outputs.
_expected_conf_df_columns = {
    "support_volume_score",
    "support_breadth_score",
    "topic_quality_score",
    "bridge_strength_score",
    "verification_cleanliness_score",
    "confidence_score",
    "confidence_band",
    "confidence_rationale",
    "supporting_project_count",
    "verified_topic_count",
    "claimed_topic_count",
    "mean_topic_share_all_verified_topics",
    "verified_topic_coherence_flags",
    "insight_id",
    "title",
    "confidence_version",
}

_actual_cols = set(conf_df.columns)
_missing = _expected_conf_df_columns - _actual_cols
_extra = _actual_cols - _expected_conf_df_columns

assert not _missing, f"CHANGE VERIFICATION FAILED — columns missing from conf_df: {_missing}"
assert not _extra, f"CHANGE VERIFICATION FAILED — unexpected new columns in conf_df: {_extra}"

assert len(conf_df) == len(candidate_evidence_rows), (
    f"CHANGE VERIFICATION FAILED — conf_df row count {len(conf_df)} "
    f"does not match candidate_evidence_rows count {len(candidate_evidence_rows)}"
)

assert conf_df["confidence_version"].eq("v1").all(), (
    "CHANGE VERIFICATION FAILED — confidence_version is not 'v1' for all rows"
)

assert conf_df["confidence_rationale"].notna().all() or rationale_map, (
    "CHANGE VERIFICATION FAILED — confidence_rationale column is entirely null "
    "and rationale_map is empty; parallel block may not have executed"
)

_rationale_none_count = conf_df["confidence_rationale"].isna().sum()
if _rationale_none_count > 0:
    print(
        f"WARNING: {_rationale_none_count} insight(s) have null confidence_rationale "
        "due to API failures — check warnings_03.jsonl for RATIONALE_API_FAILURE entries"
    )

assert set(conf_df["insight_id"]) == {row["insight_id"] for row in candidate_evidence_rows}, (
    "CHANGE VERIFICATION FAILED — insight_id set in conf_df does not match candidate_evidence_rows"
)

_score_cols = [
    "support_volume_score",
    "support_breadth_score",
    "topic_quality_score",
    "bridge_strength_score",
    "verification_cleanliness_score",
]
for col in _score_cols:
    assert conf_df[col].between(0.0, 100.0).all(), (
        f"CHANGE VERIFICATION FAILED — {col} contains values outside [0, 100]"
    )

assert conf_df["confidence_score"].between(0.0, 100.0).all(), (
    "CHANGE VERIFICATION FAILED — confidence_score contains values outside [0, 100]"
)

assert conf_df["confidence_band"].isin({"high", "medium", "low"}).all(), (
    "CHANGE VERIFICATION FAILED — confidence_band contains unexpected values: "
    f"{conf_df['confidence_band'].unique().tolist()}"
)

_conf_output_path = OUT("insights", "insights_confidence.csv")
assert _conf_output_path.exists(), (
    f"CHANGE VERIFICATION FAILED — insights_confidence.csv was not written to {_conf_output_path}"
)

print(
    "Change verification passed — conf_df structure, row count, score ranges, "
    "bands, insight_id alignment, and output file all confirmed."
)

Effective support_volume_scale_effective: 396 (multiplier=0.25, median_group_size=1586.0)
Scored 45 candidate insights
Change verification passed — conf_df structure, row count, score ranges, bands, insight_id alignment, and output file all confirmed.


---

In [15]:
# ── CONFIDENCE CALIBRATION DIAGNOSTIC ──────────────────────────────────────
# Read-only diagnostic to judge whether confidence and tiering thresholds
# will meaningfully discriminate on this specific corpus.

conf_cfg = dict(CONFIDENCE_CFG)
tier_cfg = CFG["analysis"]["tiering"]

support_volume_scale_meta = get_effective_support_volume_scale(
    df=df,
    groupby_field=GROUPBY_FIELD,
    confidence_cfg=conf_cfg,
)

conf_cfg["support_volume_scale_effective"] = support_volume_scale_meta["effective_scale"]

print(
    f"support_volume_scale_effective={conf_cfg['support_volume_scale_effective']} "
    f"(multiplier={support_volume_scale_meta['scale_multiplier']}, "
    f"median_group_size={support_volume_scale_meta['median_group_size']:.1f})"
)

diag = conf_df.copy()
diag["simulated_tier"] = diag.apply(lambda row: _simulate_tier(row, tier_cfg), axis=1)

print("\nSimulated tier counts:")
print(diag["simulated_tier"].value_counts(dropna=False).sort_index())

print("\nConfidence score distribution:")
print(diag["confidence_score"].describe())

print("\nSupporting project count distribution:")
print(diag["supporting_project_count"].describe())

print("\nVerified topic count distribution:")
print(diag["verified_topic_count"].describe())

print("\nTopic quality score distribution:")
print(diag["topic_quality_score"].describe())

print("\nBridge strength score distribution:")
print(diag["bridge_strength_score"].describe())

diag.head(10)

support_volume_scale_effective=396 (multiplier=0.25, median_group_size=1586.0)

Simulated tier counts:
simulated_tier
rejected      4
secondary    34
tertiary      3
top           4
Name: count, dtype: int64

Confidence score distribution:
count    45.000000
mean     74.282222
std      23.259898
min       0.000000
25%      66.700000
50%      78.900000
75%      88.100000
max      96.200000
Name: confidence_score, dtype: float64

Supporting project count distribution:
count      45.000000
mean      342.933333
std       333.355761
min         0.000000
25%        85.000000
50%       211.000000
75%       396.000000
max      1237.000000
Name: supporting_project_count, dtype: float64

Verified topic count distribution:
count    45.000000
mean      3.244444
std       1.667273
min       0.000000
25%       2.000000
50%       3.000000
75%       4.000000
max       8.000000
Name: verified_topic_count, dtype: float64

Topic quality score distribution:
count     45.000000
mean      90.595556
std     

,support_volume_score,support_breadth_score,topic_quality_score,bridge_strength_score,verification_cleanliness_score,confidence_score,confidence_band,confidence_rationale,supporting_project_count,verified_topic_count,claimed_topic_count,mean_topic_share_all_verified_topics,verified_topic_coherence_flags,insight_id,title,confidence_version,simulated_tier
0,100.0,100.0,100.0,77.3,100.0,94.3,high,High confidence (94.3) is driven by strong evi...,1237,8,8,0.4640,"[coherent, coherent, coherent, coherent, coher...",KI_001,Teachers treat movement and emotional regulati...,v1,secondary
1,41.7,100.0,100.0,82.6,14.1,65.3,medium,Medium confidence (65.3) is driven by strong e...,165,3,8,0.4956,"[coherent, coherent, coherent]",KI_002,Literacy demand has split into two different m...,v1,secondary
2,100.0,100.0,100.0,68.1,73.5,88.1,high,High-confidence drivers are strong evidence ac...,782,6,7,0.4088,"[coherent, coherent, coherent, coherent, coher...",KI_003,Device access is no longer a single category; ...,v1,secondary
3,73.7,100.0,100.0,95.7,64.0,85.6,high,High confidence is driven by strong breadth/qu...,292,4,5,0.5739,"[coherent, coherent, coherent, coherent]",KI_004,Gift cards have become a classroom procurement...,v1,top
4,100.0,100.0,100.0,75.4,100.0,93.9,high,High confidence is driven by strong multi-sour...,626,6,6,0.4526,"[coherent, coherent, coherent, coherent, coher...",KI_005,Color printing has become teaching infrastruct...,v1,secondary
5,89.6,100.0,100.0,77.0,36.0,81.6,high,High confidence is driven by strong support vo...,355,3,5,0.4623,"[coherent, coherent, coherent]",KI_006,Dry-erase materials are being used as a low-co...,v1,secondary
6,100.0,100.0,100.0,76.5,100.0,94.1,high,High confidence is driven by strong cross-sour...,995,7,7,0.4590,"[coherent, coherent, coherent, coherent, coher...",KI_007,Calm corners have emerged as a standard piece ...,v1,secondary
7,90.4,100.0,100.0,77.4,69.4,86.9,high,High support (volume/breadth/topic quality) an...,358,5,6,0.4643,"[coherent, coherent, coherent, coherent, coher...",KI_008,Teachers increasingly pitch invisible operatin...,v1,secondary
8,50.5,100.0,100.0,81.4,100.0,80.5,high,High confidence is driven by exceptionally str...,200,4,4,0.4882,"[coherent, coherent, coherent, coherent]",KI_009,Teachers are making ordinary purchases feel fu...,v1,top
9,52.0,100.0,100.0,88.3,69.4,78.1,medium,High support breadth (100) and strong bridge s...,206,5,6,0.5297,"[coherent, coherent, coherent, coherent, coher...",KI_010,Basic-needs requests are becoming instructiona...,v1,top


---

In [16]:
# ── TIERING ────────────────────────────────────────────────────────────────
# Apply the configured tier thresholds, then flatten all insight-level metadata.

TIER_CFG = CFG["analysis"]["tiering"]

tier_rows = []
for _, row in conf_df.iterrows():
    tier, rejected_reason_code = assign_insight_tier(row.to_dict(), TIER_CFG)
    tier_rows.append(
        {
            "insight_id": row["insight_id"],
            "tier": tier,
            "rejected_reason_code": rejected_reason_code,
            "tier_version": "v1",
        }
    )
tier_df = pd.DataFrame(tier_rows)

run_warnings_count = get_warning_count(WARNINGS_PATH)

# Build one flat output row per candidate so downstream dedupe/output logic can stay simple.
flat_rows = []
for ev in candidate_evidence_rows:
    conf = conf_df[conf_df["insight_id"] == ev["insight_id"]].iloc[0].to_dict()
    tier = tier_df[tier_df["insight_id"] == ev["insight_id"]].iloc[0].to_dict()
    flat_rows.append(
        {
            "run_id": RUN_ID,
            "insight_id": ev["insight_id"],
            "section": ev["section"],
            "category_bucket": ev["category_bucket"],
            "title": ev["title"],
            "what_seeing": ev["what_seeing"],
            "why_easy_to_miss": ev["why_easy_to_miss"],
            "source_topics_verified": ev["source_topics_verified"],
            "confidence_score": conf["confidence_score"],
            "confidence_band": conf["confidence_band"],
            "confidence_version": conf["confidence_version"],
            "confidence_rationale": conf["confidence_rationale"],
            "tier": tier["tier"],
            "tier_version": tier["tier_version"],
            "rejected_reason_code": tier["rejected_reason_code"],
            "supporting_project_count": conf["supporting_project_count"],
            "verified_topic_count": conf["verified_topic_count"],
            "source_topic_count_claimed": conf["claimed_topic_count"],
            "source_topic_count_verified": conf["verified_topic_count"],
            "group_by_field": GROUPBY_FIELD,
            "filter_logic": FILTER_LOGIC,
            "warnings_count": run_warnings_count,
        }
    )

insights_flat_df = pd.DataFrame(flat_rows)

print("Tier counts:")
print(tier_df["tier"].value_counts(dropna=False).sort_index())

Tier counts:
tier
rejected      4
secondary    34
tertiary      3
top           4
Name: count, dtype: int64


In [17]:
# ── DEDUPING ───────────────────────────────────────────────────────────────
# Remove weak one-topic items first, then screen likely overlaps and ask nano only
# on plausible duplicates/restatements.

working_df = insights_flat_df[insights_flat_df["tier"] != "rejected"].copy()

drop_one_topic_mask = working_df["verified_topic_count"].fillna(0).astype(int) < 2
dropped_one_topic_df = working_df[drop_one_topic_mask].copy()
working_df = working_df[~drop_one_topic_mask].copy()

SYSTEM_DEDUP = """
You are deduplicating insights for an external education trends report.

Insight A is already selected and is higher-priority.
Decide whether Insight B should be dropped.

Return:
- "drop_b" if Insight B is redundant with Insight A
- "keep_b" if Insight B adds a materially distinct mechanism, population, local condition, or implication

Rules:
- If both are key insights, drop B only if they would feel repetitive to a reader.
- If both are by-group insights from the same group, drop B only if it is substantially the same idea.
- If one is a key insight and the other is a by-group insight, keep the by-group insight if it adds a genuinely local mechanism or condition. Drop it if it mostly restates the key insight.

Be conservative.
If B adds clear local specificity or a different operating logic, keep it.

Return valid JSON only:
{"decision":"drop_b"|"keep_b","reason":"..."}
""".strip()

# Precompute lightweight fields used in the heuristic screen before any model call.
working_df["verified_topics_list"] = working_df["source_topics_verified"].apply(_topic_list)
working_df["title_tokens"] = working_df["title"].apply(_token_set)
working_df["text_tokens"] = (
    working_df["title"].fillna("") + " "
    + working_df["what_seeing"].fillna("") + " "
).apply(_token_set)

# Strongest insights go first so lower-signal variants are compared against them.
working_df = working_df.sort_values(
    ["confidence_score", "verified_topic_count", "supporting_project_count"],
    ascending=False,
).reset_index(drop=True)

kept_rows = []
dropped_dedup_ids = set()
dedup_audit_rows = []
nano_review_count = 0

for _, row in working_df.iterrows():
    drop_current = False

    for kept in kept_rows:
        screen_meta = _screen_pair(row, kept)
        if screen_meta is None:
            continue

        nano_review_count += 1
        decision = _nano_dedup_decision(
            kept,
            row,
            screen_meta,
            client=client,
            model_name=MODEL_DEDUP,
            system_prompt=SYSTEM_DEDUP,
        )

        if decision.get("decision") == "drop_b":
            drop_current = True
            dropped_dedup_ids.add(row["insight_id"])
            dedup_audit_rows.append(
                {
                    "dropped_insight_id": row["insight_id"],
                    "matched_kept_insight_id": kept["insight_id"],
                    "pair_kind": screen_meta["pair_kind"],
                    "topic_overlap": screen_meta["topic_overlap"],
                    "title_overlap": screen_meta["title_overlap"],
                    "text_overlap": screen_meta["text_overlap"],
                    "reason": decision.get("reason", ""),
                }
            )
            break

    if not drop_current:
        kept_rows.append(row.to_dict())

curated_df = pd.DataFrame(kept_rows).copy()
dropped_dedup_df = working_df[working_df["insight_id"].isin(dropped_dedup_ids)].copy()

print(f"Non-rejected before cleanup: {len(insights_flat_df[insights_flat_df['tier'] != 'rejected'])}")
print(f"Dropped for verified_topic_count < 2: {len(dropped_one_topic_df)}")
print(f"Nano reviews run for dedupe: {nano_review_count}")
print(f"Dropped as duplicates/restatements: {len(dropped_dedup_df)}")
print(f"Accepted after cleanup: {len(curated_df)}")

accepted_ids = set(curated_df["insight_id"])
rejected_ids = set(insights_flat_df["insight_id"]) - accepted_ids

accepted_evidence_rows = [
    row for row in candidate_evidence_rows if row["insight_id"] in accepted_ids
]
rejected_evidence_rows = [
    row for row in candidate_evidence_rows if row["insight_id"] in rejected_ids
]

Non-rejected before cleanup: 41
Dropped for verified_topic_count < 2: 1
Nano reviews run for dedupe: 5
Dropped as duplicates/restatements: 1
Accepted after cleanup: 39


In [18]:
# ── FINAL OUTPUTS + REPORTING ──────────────────────────────────────────────
# Persist accepted/rejected outputs, evidence exports, chart data, DOCX, and manifests.

insights_flat_df.to_csv(OUT("insights", "insights_flat.csv"), index=False)

write_json(
    OUT("insights", "rejected_insights.json"),
    [
        {
            **row,
            "confidence": conf_df[conf_df["insight_id"] == row["insight_id"]].iloc[0].to_dict(),
            "tiering": tier_df[tier_df["insight_id"] == row["insight_id"]].iloc[0].to_dict(),
        }
        for row in rejected_evidence_rows
    ],
)

structured = {"key_insights": [], OUTPUT_GROUP_KEY: {}}
for candidate in candidate_rows:
    ev = next(
        (r for r in candidate_evidence_rows if r["insight_id"] == candidate["insight_id"]),
        None,
    )
    if candidate["insight_id"] not in accepted_ids or ev is None:
        continue

    insight_clean = {
        "insight_id": candidate["insight_id"],
        "title": ev["title"],
        "what_seeing": ev["what_seeing"],
        "why_easy_to_miss": ev["why_easy_to_miss"],
        "source_topics": ev["source_topics_verified"],
        "confidence_score": float(
            conf_df.loc[
                conf_df["insight_id"] == candidate["insight_id"],
                "confidence_score",
            ].iloc[0]
        ),
        "confidence_band": conf_df.loc[
            conf_df["insight_id"] == candidate["insight_id"],
            "confidence_band",
        ].iloc[0],
        "confidence_version": "v1",
        "tier": tier_df.loc[
            tier_df["insight_id"] == candidate["insight_id"],
            "tier",
        ].iloc[0],
        "tier_version": "v1",
        "supporting_project_count": int(ev["supporting_project_count"]),
        "verified_topic_count": int(len(ev["source_topics_verified"])),
        "warnings": [],
    }

    if candidate["section"] == "key_insights":
        structured["key_insights"].append(insight_clean)
    else:
        structured[OUTPUT_GROUP_KEY].setdefault(candidate["category_bucket"], []).append(insight_clean)

write_json(OUT("insights", "insights_structured.json"), structured)

# Accepted-only evidence artifacts.
with open(OUT("evidence", "evidence_pack.jsonl"), "w", encoding="utf-8") as f:
    for row in accepted_evidence_rows:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

evidence_topic_support_rows = []
evidence_snippet_rows = []
evidence_project_rows = []

for row in accepted_evidence_rows:
    topic_rows_cache = {
        get_topic_key(GROUPBY_FIELD, ts["group"], ts["topic_id"]): BRIDGE_LOOKUP.get(
            get_topic_key(GROUPBY_FIELD, ts["group"], ts["topic_id"]),
            pd.DataFrame(columns=["project_id", "weight", "topic_share"]),
        )
        for ts in row["topic_support"]
    }

    for ts in row["topic_support"]:
        evidence_topic_support_rows.append(
            {
                "run_id": RUN_ID,
                "insight_id": row["insight_id"],
                "section": row["section"],
                "category_bucket": row["category_bucket"],
                "group_by_field": GROUPBY_FIELD,
                "group_value": ts["group"],
                "topic_id": ts["topic_id"],
                "topic_label": ts["topic_label"],
                "topic_description": ts["topic_description"],
                "coherence_flag": ts["coherence_flag"],
                "supporting_project_count": ts["supporting_project_count"],
                "mean_topic_share": ts["mean_topic_share"],
                "median_topic_share": ts["median_topic_share"],
            }
        )

        topic_key = get_topic_key(GROUPBY_FIELD, ts["group"], ts["topic_id"])
        bridge_rows = topic_rows_cache.get(
            topic_key,
            pd.DataFrame(columns=["project_id", "weight", "topic_share"]),
        )
        for _, br in bridge_rows.iterrows():
            evidence_project_rows.append(
                {
                    "run_id": RUN_ID,
                    "insight_id": row["insight_id"],
                    "group_by_field": GROUPBY_FIELD,
                    "group_value": ts["group"],
                    "topic_id": ts["topic_id"],
                    "project_id": br["project_id"],
                    "topic_key": topic_key,
                    "weight": br["weight"],
                    "topic_share": br["topic_share"],
                }
            )

    for sp in row["sample_projects"]:
        evidence_snippet_rows.append(
            {
                "run_id": RUN_ID,
                "insight_id": row["insight_id"],
                "group_by_field": GROUPBY_FIELD,
                "group_value": sp["group"],
                "topic_id": sp["topic_id"],
                "project_id": sp["project_id"],
                "topic_share": sp["topic_share"],
                "essay_text": build_full_essay_text(sp["project_id"], essay_lookup),
                "injected_tokens": build_injected_tokens(sp["project_id"], token_lookup),
            }
        )

pd.DataFrame(evidence_topic_support_rows).to_csv(
    OUT("evidence", "evidence_topic_support.csv"),
    index=False,
)
pd.DataFrame(evidence_snippet_rows).to_csv(
    OUT("evidence", "evidence_snippets.csv"),
    index=False,
)
pd.DataFrame(evidence_project_rows).to_csv(
    OUT("evidence", "evidence_projects.csv"),
    index=False,
)

# Chart-ready accepted-only data.
chart_rows = []
group_support_rows = []
for row in accepted_evidence_rows:
    flat = insights_flat_df[insights_flat_df["insight_id"] == row["insight_id"]].iloc[0]
    chart_rows.append(
        {
            "run_id": RUN_ID,
            "insight_id": row["insight_id"],
            "theme": row["title"],
            "section": row["section"],
            "category_bucket": row["category_bucket"],
            "tier": flat["tier"],
            "confidence_band": flat["confidence_band"],
            "confidence_score": flat["confidence_score"],
            "group_by_field": GROUPBY_FIELD,
            "supporting_project_count": row["supporting_project_count"],
            "verified_topic_count": len(row["source_topics_verified"]),
            "magnitude": row["supporting_project_count"],
            "magnitude_unit": "projects",
            "delta": None,
            "delta_unit": None,
            "period": None,
        }
    )
    for ts in row["topic_support"]:
        group_support_rows.append(
            {
                "run_id": RUN_ID,
                "insight_id": row["insight_id"],
                "theme": row["title"],
                "group_by_field": GROUPBY_FIELD,
                "group_value": ts["group"],
                "topic_id": ts["topic_id"],
                "supporting_project_count": ts["supporting_project_count"],
                "mean_topic_share": ts["mean_topic_share"],
                "confidence_score": flat["confidence_score"],
                "confidence_band": flat["confidence_band"],
                "tier": flat["tier"],
            }
        )

pd.DataFrame(chart_rows).to_csv(OUT("chart_data", "chart_ready_insights.csv"), index=False)
pd.DataFrame(group_support_rows).to_csv(
    OUT("chart_data", "chart_ready_group_support.csv"),
    index=False,
)

# DOCX report for accepted insights only.
docx_path = OUT("reports", "trend_tracker_report.docx")
build_trend_tracker_report_docx(
    structured=structured,
    output_path=docx_path,
    groupby_field=GROUPBY_FIELD,
    output_group_key=OUTPUT_GROUP_KEY,
    report_cfg=CFG["output"],
    project_count=len(df),
)

# Metadata / manifests.
write_json(
    OUT("metadata", "confidence_model_meta.json"),
    {
        "schema_version": "v1",
        "run_id": RUN_ID,
        "group_by_field": GROUPBY_FIELD,
        "filter_fields_key": FILTER_FIELDS_KEY,
        "confidence_config": CONFIDENCE_CFG,
    },
)
write_json(
    OUT("metadata", "tiering_model_meta.json"),
    {
        "schema_version": "v1",
        "run_id": RUN_ID,
        "group_by_field": GROUPBY_FIELD,
        "filter_fields_key": FILTER_FIELDS_KEY,
        "tiering_config": TIER_CFG,
    },
)

groups_failed = sorted(set(NMF_GROUPS_FAILED) | set(LABELING_FAILED_GROUPS) | set(SYNTHESIS_FAILED_GROUPS))
eligible_groups = list(per_group_results.keys())

stage_manifest_path = OUT("metadata", "stage_manifest_03_insights_generation.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status="success",
    input_artifacts=[
        artifact_meta(ROOT / "OUTPUTS/prepared/05_enriched.parquet", "enriched_parquet"),
        artifact_meta(ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json", "stage_manifest_01"),
        artifact_meta(ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json", "stage_manifest_02"),
    ],
    output_artifacts=[
        artifact_meta(COPIED_CONFIG_PATH, "resolved_params_yaml"),
        artifact_meta(OUT("analysis", "category_tfidf.csv"), "category_tfidf_csv"),
        artifact_meta(OUT("analysis", "nmf_topics.csv"), "nmf_topics_csv"),
        artifact_meta(OUT("analysis", "nmf_weights.csv"), "nmf_weights_csv"),
        artifact_meta(OUT("analysis", "project_topic_bridge.csv"), "project_topic_bridge_csv"),
        artifact_meta(OUT("analysis", "llm_topic_labels.json"), "llm_topic_labels_json"),
        artifact_meta(OUT("insights", "insights_candidates.json"), "insights_candidates_json"),
        artifact_meta(OUT("insights", "insights_structured.json"), "insights_structured_json"),
        artifact_meta(OUT("insights", "rejected_insights.json"), "rejected_insights_json"),
        artifact_meta(OUT("insights", "insights_flat.csv"), "insights_flat_csv"),
        artifact_meta(OUT("insights", "insights_confidence.csv"), "insights_confidence_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_insights.csv"), "chart_ready_insights_csv"),
        artifact_meta(OUT("reports", "trend_tracker_report.docx"), "trend_tracker_report_docx"),
    ],
    row_counts={
        "input_projects": int(len(raw_df)),
        "filtered_projects": int(len(df)),
        "eligible_groups": int(len(eligible_groups)),
        "groups_skipped": int(len(NMF_GROUPS_SKIPPED)),
        "groups_failed": int(len(groups_failed)),
        "topics_generated": int(len(topics_df)),
        "insights_generated": int(len(candidate_evidence_rows)),
        "insights_accepted": int(len(accepted_ids)),
        "insights_rejected": int(len(rejected_ids)),
    },
    key_params={
        "group_by": GROUPBY_FIELD,
        "filter_fields_key": FILTER_FIELDS_KEY,
        "min_group_projects": CFG["analysis"]["min_group_projects"],
        "topic_assignment_threshold": CFG["analysis"]["topic_assignment_threshold"],
        "model_labeling": MODEL_LABELING,
        "model_synthesis": MODEL_SYNTHESIS,
        "confidence_config": CONFIDENCE_CFG,
        "tiering_config": TIER_CFG,
    },
    warnings_path=WARNINGS_PATH,
)

pipeline_manifest_path = OUT("metadata", "pipeline_manifest.json")
build_pipeline_manifest(
    output_path=pipeline_manifest_path,
    run_id=RUN_ID,
    run_date=RUN_DATE,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
    filter_spec_path=FILTER_SPEC_PATH,
    filter_summary_path=FILTER_SUMMARY_PATH,
    stage_manifest_paths=[
        ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json",
        ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json",
        stage_manifest_path,
    ],
    warnings_01_path=ROOT / "OUTPUTS/prepared/warnings_01.jsonl",
    warnings_02_path=ROOT / "OUTPUTS/enrichment/warnings_02.jsonl",
    warnings_03_path=WARNINGS_PATH,
    final_outputs={
        "insights_structured_json": str(OUT("insights", "insights_structured.json")),
        "insights_flat_csv": str(OUT("insights", "insights_flat.csv")),
        "evidence_pack_jsonl": str(OUT("evidence", "evidence_pack.jsonl")),
        "insights_confidence_csv": str(OUT("insights", "insights_confidence.csv")),
        "chart_ready_insights_csv": str(OUT("chart_data", "chart_ready_insights.csv")),
        "trend_tracker_report_docx": str(OUT("reports", "trend_tracker_report.docx")),
    },
)

print(f"Accepted insights: {len(accepted_ids):,}")
print(f"Rejected insights: {len(rejected_ids):,}")
print(f"Docx saved → {docx_path}")
print(f"Stage manifest written → {stage_manifest_path}")
print(f"Pipeline manifest written → {pipeline_manifest_path}")

Accepted insights: 39
Rejected insights: 6
Docx saved → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/miami_geo_group/2026-04-02/20260402_200024_miami_geo_group_438c7b3e/reports/trend_tracker_report.docx
Stage manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/miami_geo_group/2026-04-02/20260402_200024_miami_geo_group_438c7b3e/metadata/stage_manifest_03_insights_generation.json
Pipeline manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/miami_geo_group/2026-04-02/20260402_200024_miami_geo_group_438c7b3e/metadata/pipeline_manifest.json
